In [1]:
import pandas as pd

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
meta = pd.read_excel('liam_basal_metadata.xlsx')
meta.head()

,core,time,section,face,ACorDC,AC_col_sp,DC_col_sp,DC_volt,note,mm_per_encode_step,...,DC_edgespace,idx1_raw,idx1_rel,idx2_rel,idx3_rel,idx_abs,xmin,xmax,header,filename
0,dic1,2023-05-25-07-29,222a,t,DC,10,25,NaN,first run of day,0.25,...,25,NaN,0,NaN,NaN,203.374,NaN,NaN,22,2023-05-25-07-29-DIC1-222.txt
1,dic1,2023-05-25-07-52,222a,t,AC,10,25,NaN,try again on AC,0.25,...,25,NaN,0,NaN,NaN,203.374,NaN,NaN,22,2023-05-25-07-52-DIC1-222.txt
2,dic1,2023-05-25-08-17,222b,t,AC,10,25,NaN,seccond peice of dic1-222,0.25,...,25,NaN,379,NaN,NaN,203.753,NaN,NaN,22,2023-05-25-08-17-dic1-222b.txt
3,dic1,2023-05-25-08-37,222b,t,DC,10,25,NaN,dc run on 2nd peice of dic1-222,0.25,...,25,NaN,379,NaN,NaN,203.753,NaN,NaN,22,2023-05-25-08-37-dic1-222b.txt
4,dic1,2023-05-25-09-14,223e,t,DC,10,25,NaN,fifth piece of 223e - dc,0.25,...,25,NaN,555,NaN,NaN,204.666,NaN,NaN,22,2023-05-25-09-14-dic1-223e.txt


In [4]:
for idx, row in meta.iterrows():

    # load df
    df = pd.read_csv(str(row['section'])+
                     '_'+row['ACorDC']+
                     '_'+row['time']+
                     '.csv',
                     header = None)
    
    # name columns
    df.columns = ['raw_depth','True_depth(m)','Y_dimension(mm)','meas','Button']
    df = df.drop(columns=['raw_depth'])

    # sort column names
    df = df[['Y_dimension(mm)','Button','True_depth(m)','meas']]

    # update metadata - Y bounds
    if row['ACorDC'] == 'AC':
        edgespace = row['AC_edgespace']
    else:
        edgespace = row['DC_edgespace']
    meta.at[idx, 'Y_left'] = min(df['Y_dimension(mm)']) - edgespace
    meta.at[idx, 'Y_right'] = max(df['Y_dimension(mm)']) + edgespace

    # update metadata - filename
    fname = (row['core']+
             '-'+str(row['section'])+
             '-'+row['face']+
             '-'+row['ACorDC']+
             '.csv')
    meta.at[idx, 'filename'] = (fname)
    
    # save file
    df.to_csv('../liam_basal_data/'+fname, index=False)

# save updated metadata
meta.to_csv('../liam_basal_data/metadata_liam_basal.csv', index=False)
    